# Lab 3: Circular Queues and Singly Linked Lists

**Z5007 Programming and Data Structures · Lab 3 · Thursday 29 October 2026, 3:25–4:40 pm**

Today you build, yourself, the two structures from Wednesday and this morning: the **circular-array queue** (Lecture 12) and the
**singly linked list** (Lecture 13). Then you put a queue on your linked list, race it, and reverse a list in place. Step 8, one
problem with no skeleton at all, is an optional take-home bonus (+10). Each passed test prints a progress line, so you always know how far you are. The tracker stops at 90: the last 10 points are the graded questions, which I mark by hand.

### The route
| Time | Step | What you do | Points |
|---|---|---|---|
| 3:25–3:30 | Step 0 | Run me first; draw three pictures on paper | 5 |
| 3:30–3:46 | Step 1 | Circular queue: `enqueue`, `dequeue`, `peek` | 20 |
| 3:46–3:53 | Step 2 | Linked list: `push_front` and `to_list` | 10 |
| 3:53–4:00 | Step 3 | Linked list: `push_back` and `pop_front` (the special cases) | 10 |
| 4:00–4:13 | Step 4 | Linked list: `find` and `remove` | 20 |
| 4:13–4:15 | Step 5 | Stress test: 5,000 random operations against a Python list (just run) | 5 |
| 4:15–4:23 | Step 6 | A queue on your linked list, and a race | 10 |
| 4:23–4:31 | Step 7 | Reverse a linked list in place | 10 |
| 4:31–4:37 | Graded questions | Three short written answers | 10 |
| 4:37–4:40 | Submit | Kernel → Restart & Run All, upload to Google Classroom | – |
| After class | Step 8 (optional) | **Take-home bonus:** is a linked list a palindrome, with O(1) extra memory? | +10 bonus |

**How to work.** Every step gives a **goal**, method stubs with their **contract** (docstring), and a **test**. You write every method body
yourself. Draw boxes and arrows on paper before you code each linked-list method. If you are stuck for more than 5 minutes, read the
**Hints** at the end (one line each, no marks lost), then ask me.

**Conventions (same as the lectures).** Circular queue: `head` = index of the front item, `tail` = index of the next FREE box, `count` =
number of items; empty when `count == 0`, full when `count == capacity`. Linked list: `head` = first node, `tail` = last node, both `None`
when the list is empty.

## Step 0 (3:25–3:30). Set up, then draw

Run the cell below (click it, press **Shift + Enter**).

In [ ]:
# RUN THIS CELL FIRST: progress tracker (nothing to change)
XP = {}
def cleared(step, title, xp):
    XP[step] = xp
    total = sum(XP.values())
    bar = "#" * (total // 4) + "." * ((100 - total) // 4)
    print(f"{step} passed: {title}  (+{xp} points)")
    print(f"Progress [{bar}] {total}/100")
cleared("Step 0", "set-up", 5)

**Draw on paper before you continue (no marks, but you need these pictures for Steps 1 to 4):**
1. A circular queue of capacity 4, empty (head = tail = count = 0). Enqueue A, B, C; dequeue; enqueue D, E. Draw the array and write head, tail, count.
2. The linked list `head → 12 → 7 → 33 → None` after `push_front(5)`. Number the two pointer changes in the order you make them.
3. The same list after removing 7. Which node's `next` changes?

## Step 1 (3:30–3:46). A circular queue (20 points)

**Goal.** A queue on a fixed array where `enqueue` and `dequeue` are both O(1): **nothing ever shifts**.

Follow the conventions above (Lecture 12).

In [ ]:
class CircularQueue:
    def __init__(self, capacity):
        self.cap = capacity
        self.a = [None] * capacity
        self.head = 0          # index of the front item
        self.tail = 0          # index of the next FREE box
        self.count = 0         # number of items

    def __len__(self):
        return self.count

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.cap

    def enqueue(self, x):
        """Add x at the back in O(1). Raise OverflowError if full."""
        # YOUR CODE HERE
        raise NotImplementedError

    def dequeue(self):
        """Remove and return the front item in O(1). Raise IndexError if empty.
        The array must not keep a reference to the removed item."""
        # YOUR CODE HERE
        raise NotImplementedError

    def peek(self):
        """Return the front item without removing it. Raise IndexError if empty."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# TEST 1: do not edit
q = CircularQueue(4)
assert q.is_empty() and not q.is_full()
for x in "ABC": q.enqueue(x)
assert (q.head, q.tail, q.count) == (0, 3, 3), "after A, B, C: head 0, tail 3, count 3"
assert q.dequeue() == "A" and q.peek() == "B" and len(q) == 2
q.enqueue("D"); q.enqueue("E")                      # E goes into box 0: tail wraps
assert q.a == ["E", "B", "C", "D"], f"array should be [E, B, C, D], got {q.a}"
assert (q.head, q.tail, q.count) == (1, 1, 4), f"expected head 1, tail 1, count 4, got {q.head, q.tail, q.count}"
assert q.is_full()
try:
    q.enqueue("F"); raise AssertionError("a full queue must raise OverflowError")
except OverflowError: pass
assert [q.dequeue() for _ in range(4)] == ["B", "C", "D", "E"], "first in, first out"
assert q.a == [None] * 4, "dequeue must clear the box"
for bad in (q.dequeue, q.peek):
    try:
        bad(); raise AssertionError("an empty queue must raise IndexError")
    except IndexError: pass
import random
random.seed(1)
q, ref = CircularQueue(7), []
for _ in range(20000):                               # random operations against a Python list
    if ref and (len(ref) == 7 or random.random() < 0.5):
        assert q.dequeue() == ref.pop(0)
    else:
        v = random.randint(0, 99); q.enqueue(v); ref.append(v)
    assert len(q) == len(ref)
cleared("Step 1", "circular queue", 20)

## Step 2 (3:46–3:53). Linked list: `push_front` and `to_list` (10 points)

**Goal.** The `Node` class, the list object, a walk that collects the values, and insertion at the front in O(1).

Every method must leave `head`, `tail` and `n` correct, including on an empty list (Lecture 13).

This cell holds the whole `LinkedList` class. In Steps 2, 3 and 4 you come back to **this same cell**, fill in more methods, and run it again.

In [ ]:
class Node:
    def __init__(self, val, next=None):
        self.val = val          # the data
        self.next = next        # the next node, or None at the end


class LinkedList:
    def __init__(self):
        self.head = None        # first node (None when empty)
        self.tail = None        # last node  (None when empty)
        self.n = 0              # number of nodes

    def __len__(self):
        return self.n

    # ---------------- Step 2
    def to_list(self):
        """Return the values from head to tail as a Python list (used by the tests)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def push_front(self, x):
        """Insert x at the front in O(1)."""
        # YOUR CODE HERE
        raise NotImplementedError

    # ---------------- Step 3
    def push_back(self, x):
        """Insert x at the back in O(1)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def pop_front(self):
        """Remove and return the front value in O(1). Raise IndexError if empty."""
        # YOUR CODE HERE
        raise NotImplementedError

    # ---------------- Step 4
    def find(self, x):
        """Return True if some node holds x, else False."""
        # YOUR CODE HERE
        raise NotImplementedError

    def remove(self, x):
        """Delete the FIRST node holding x. Return True if one was removed, False if x is not in the list."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# TEST 2: do not edit
L = LinkedList()
assert L.to_list() == [] and len(L) == 0
L.push_front(33); assert L.head is L.tail, "one node: head and tail are the same node"
L.push_front(7); L.push_front(12)
assert L.to_list() == [12, 7, 33] and len(L) == 3
assert L.head.val == 12 and L.tail.val == 33 and L.tail.next is None
cleared("Step 2", "push_front and to_list", 10)

## Step 3 (3:53–4:00). Linked list: `push_back` and `pop_front` (10 points)

**Goal.** Add at the end in O(1) using `tail`; remove from the front in O(1).

Go back to the `LinkedList` cell, write the Step 3 methods, run that cell again, then run this test.

In [ ]:
# TEST 3: do not edit
L = LinkedList()
L.push_back(1); assert L.head is L.tail and L.to_list() == [1]
L.push_back(2); L.push_back(3); L.push_front(0)
assert L.to_list() == [0, 1, 2, 3] and L.tail.val == 3 and len(L) == 4
assert [L.pop_front() for _ in range(4)] == [0, 1, 2, 3]
assert L.head is None and L.tail is None, "after the last pop, BOTH head and tail must be None"
try:
    L.pop_front(); raise AssertionError("pop on an empty list must raise IndexError")
except IndexError: pass
L.push_back(9); assert L.to_list() == [9] and L.head is L.tail, "push_back after emptying the list"
cleared("Step 3", "push_back and pop_front", 10)

## Step 4 (4:00–4:13). Linked list: `find` and `remove` (20 points)

**Goal.** Search, and delete the first node holding a value.

Draw three cases before you code: deleting the first, a middle and the last node.

In [ ]:
# TEST 4: do not edit
L = LinkedList()
for x in [5, 12, 7, 33, 7]: L.push_back(x)
assert L.find(33) and not L.find(99)
assert L.remove(7) and L.to_list() == [5, 12, 33, 7], "remove only the FIRST 7"
assert L.remove(5) and L.to_list() == [12, 33, 7] and L.head.val == 12, "removing the first node"
assert L.remove(7) and L.to_list() == [12, 33] and L.tail.val == 33, "removing the last node: tail must move back"
assert not L.remove(99) and len(L) == 2
L.push_back(40); assert L.to_list() == [12, 33, 40], "push_back after removing the tail"
assert L.remove(12) and L.remove(33) and L.remove(40)
assert L.head is None and L.tail is None and len(L) == 0, "removing every node empties the list"
cleared("Step 4", "find and remove", 20)

## Step 5 (4:13–4:15). Stress test (5 points)

Just run it. Your list is compared with a Python list on 5,000 random operations, and `head`, `tail` and `len` are checked after every one.
If it fails, the message says which operation broke; draw that case on paper.

In [ ]:
# TEST 5: do not edit
import random
random.seed(4)
L, ref = LinkedList(), []
for step in range(5000):
    op = random.choice(["front", "back", "pop", "remove", "find"])
    x = random.randint(0, 20)
    if op == "front": L.push_front(x); ref.insert(0, x)
    elif op == "back": L.push_back(x); ref.append(x)
    elif op == "pop":
        if ref: assert L.pop_front() == ref.pop(0), f"step {step}: pop_front"
        else:
            try: L.pop_front(); raise AssertionError("empty pop must raise IndexError")
            except IndexError: pass
    elif op == "remove":
        assert L.remove(x) == (x in ref), f"step {step}: remove({x})"
        if x in ref: ref.remove(x)
    else:
        assert L.find(x) == (x in ref), f"step {step}: find({x})"
    assert len(L) == len(ref), f"step {step} ({op}): len"
    assert (L.head.val if L.head else None) == (ref[0] if ref else None), f"step {step} ({op}): head"
    assert (L.tail.val if L.tail else None) == (ref[-1] if ref else None), f"step {step} ({op}): tail"
assert L.to_list() == ref
cleared("Step 5", "stress test", 5)

## Step 6 (4:15–4:23). A queue on your linked list, and a race (10 points)

**Goal.** A queue with no capacity limit, built on your `LinkedList`, with O(1) `enqueue` and `dequeue`.

Use only your `LinkedList` methods; write no new pointer code.

In [ ]:
class LinkedQueue:
    def __init__(self):
        self.items = LinkedList()

    def __len__(self):
        return len(self.items)

    def enqueue(self, x):
        """Add x at the back in O(1)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def dequeue(self):
        """Remove and return the front item in O(1)."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# TEST 6 and the race: do not edit
q = LinkedQueue()
for x in "abc": q.enqueue(x)
assert q.dequeue() == "a" and len(q) == 2 and q.dequeue() == "b"
import time
from collections import deque
N = 20_000
def race_list():
    a = list(range(N))
    for i in range(N): a.append(i); a.pop(0)
def race_linked():
    q = LinkedQueue()
    for i in range(N): q.enqueue(i)
    for i in range(N): q.enqueue(i); q.dequeue()
def race_deque():
    d = deque(range(N))
    for i in range(N): d.append(i); d.popleft()
for name, f in [("list.pop(0)", race_list), ("LinkedQueue", race_linked), ("deque", race_deque)]:
    t = time.perf_counter(); f(); print(f"{name:<12} {time.perf_counter() - t:.3f} s")
cleared("Step 6", "linked queue", 10)

## Step 7 (4:23–4:31). Reverse a linked list in place (10 points)

**Goal.** Turn every arrow round, creating **no new nodes**, in O(n) time and O(1) extra memory.

Lecture 13, last technique. `head` and `tail` must be correct afterwards.

In [ ]:
def reverse(L):
    """Reverse the LinkedList L in place: no new nodes, O(n) time, O(1) extra memory. Return nothing."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# TEST 7: do not edit
def node_ids(L):
    out, cur = [], L.head
    while cur is not None:
        out.append(id(cur)); cur = cur.next
    return sorted(out)
for vals in ([], [1], [1, 2], [3, 1, 4, 1, 5, 9]):
    L = LinkedList()
    for v in vals: L.push_back(v)
    ids = node_ids(L)
    reverse(L)
    assert L.to_list() == vals[::-1], f"reverse of {vals}"
    assert (L.tail.val if L.tail else None) == (vals[0] if vals else None) and (L.tail is None or L.tail.next is None)
    assert ids == node_ids(L), "reverse must reuse the same nodes, not create new ones"
L = LinkedList()
for v in range(200_000): L.push_back(v)
t = time.perf_counter(); reverse(L); dt = time.perf_counter() - t
assert L.head.val == 199_999 and dt < 2.0, "200,000 nodes should reverse in well under a second"
cleared("Step 7", "reverse in place", 10)

## Graded questions (4:31–4:37, 10 points, marked by hand)

Answer in the cell below. Two to four sentences each; an answer without a reason scores zero.

**G1 (3 points).** In your `CircularQueue`, after some operations `head == tail`. Give one sequence of operations on a capacity-3 queue that
ends with `head == tail` and the queue **empty**, and one that ends with `head == tail` and the queue **full**. Why does the class need `count`?

**G2 (4 points).** Report your three race times from Step 6 and rank them. Explain the ranking with Big-O for `list.pop(0)` and for your
`LinkedQueue`, and give one practical reason why `deque` is faster than your `LinkedQueue` although both are O(1).

**G3 (3 points).** Your `LinkedList` keeps `tail`, so `push_back` is O(1). Explain why a `pop_back` (remove the last node) would still be
O(n). What change to the node would make it O(1)? (This is Friday's lecture.)

*Your answers to G1, G2 and G3 here.*

## Step 8 (optional, take-home bonus). On your own: is a linked list a palindrome? (+10 bonus points)

No skeleton this time. Write `is_palindrome(L)`, which returns `True` if the values of the `LinkedList` `L` read the same forwards and
backwards (`1 → 2 → 2 → 1` and `1 → 2 → 1` are palindromes; `1 → 2` is not; the empty list and a one-node list are).

**Rules.** O(n) time and **O(1) extra memory**: do not copy the values into a Python list, string, stack or any other container, and do not
call `L.to_list()`. You may change the links temporarily, but when the function returns, **`L` must be exactly as it was** (same order, same
`head` and `tail`). Everything you need was in this morning's lecture. Plan on paper first: which two techniques do you combine?

Below your code, write 3–5 sentences: your plan, and why the extra memory is O(1).

In [ ]:
def is_palindrome(L):
    # YOUR CODE HERE (no skeleton)
    raise NotImplementedError

*Your plan and the memory argument here.*

In [ ]:
# TEST 8: do not edit
import tracemalloc
def make(vals):
    L = LinkedList()
    for v in vals: L.push_back(v)
    return L
for vals, want in [([], True), ([7], True), ([1, 2], False), ([4, 4], True), ([1, 2, 1], True), ([1, 2, 2, 1], True),
                   ([1, 2, 3], False), ([1, 2, 3, 2, 2], False), (list("RACECAR"), True), ([1, 2, 3, 3, 2, 9], False)]:
    L = make(vals); h, t = L.head, L.tail
    L.to_list = lambda: (_ for _ in ()).throw(AssertionError("do not call to_list()"))
    got = is_palindrome(L)
    assert got == want, f"{vals}: expected {want}, got {got}"
    del L.to_list
    assert L.to_list() == vals and L.head is h and L.tail is t and (t is None or t.next is None), f"{vals}: the list was not restored"
big = list(range(100_000)) + list(range(100_000))[::-1]
L = make(big)
tracemalloc.start(); r = is_palindrome(L); peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
assert r is True and L.to_list() == big
assert peak < 100_000, f"extra memory {peak:,} bytes for 200,000 nodes: values were copied into a container"
cleared("Step 8", "palindrome with O(1) extra memory (bonus)", 10)

## Hints (only if you are stuck for 5 minutes)
* **Step 1.** What happens to an index after the last box?
* **Step 3.** After the last `pop_front`, where does `tail` point?
* **Step 4.** To unlink a node, whose `next` must change?
* **Step 6.** Which end of a singly linked list is O(1) to remove from?
* **Step 7.** What do you lose when you overwrite `cur.next`?
* **Step 8.** Two techniques from this morning; one of them is Step 7.

## Before you submit (4:37–4:40)
* Every test prints `passed` (Step 8 only if attempted). G1, G2 and G3 are written (and the Step 8 explanation, if you did the bonus).
* *Kernel → Restart & Run All* works without errors. Then upload this notebook to Google Classroom.
* If you used an AI tool, add one line here saying what for.